# 3. Grouped nested model evaluation

Seven classifiers × three representations; 5 outer folds × 3 repeats, 4 inner folds, 24 search trials.

## Setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "mea_ml" / "io.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

import json
import hashlib
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from mea_ml.io import ROOT as PROJECT_ROOT, training_protocol as make_training_protocol
from mea_ml.training import (
    RANDOM_STATE, OUTER_N_SPLITS, N_REPEATS, OUTER_SEEDS, BAYES_N_INIT, BAYES_N_ITER,
    make_outer_folds, make_inner_folds, make_model_specs, subset_rows,
    bayes_optimize_model, fit_predict_one_split, compute_metrics,
)

OUT_DIR = PROJECT_ROOT / "results/03_ml"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = PROJECT_ROOT / "results/01_dataset/training_bundle.joblib"
bundle = joblib.load(BUNDLE_PATH)


## Grouped folds

In [2]:
X_mean_delta_df = bundle["X_mean_delta_df"].copy()
X_delta_only_df = bundle["X_delta_only_df"].copy()
X_mfr_only_df = bundle["X_mfr_only_df"].copy()

feature_names_mean_delta = list(bundle["feature_names_mean_delta"])
feature_names_delta_only = list(bundle["feature_names_delta_only"])
feature_names_mfr_only = list(bundle["feature_names_mfr_only"])

y = np.asarray(bundle["y"]).astype(int)
groups = np.asarray(bundle["groups"]).astype(str)
paired_meta = bundle["paired_meta"].copy()
feature_manifest = bundle["feature_manifest"].copy()
feature_catalog = bundle.get("feature_catalog", pd.DataFrame())

assert len(X_mean_delta_df) == len(y) == len(groups) == len(paired_meta)

print("X_mean_delta:", X_mean_delta_df.shape)
print("X_delta_only:", X_delta_only_df.shape)
print("X_mfr_only:", X_mfr_only_df.shape)
print("N samples:", len(y))
print("N groups:", pd.Series(groups).nunique())

group_df = paired_meta[["group_id"]].copy()
group_df["y"] = y
group_label_df = (
    group_df.groupby("group_id", as_index=False)["y"].first()
    .sort_values(["y", "group_id"])
    .reset_index(drop=True)
)

pos_groups = group_label_df.loc[group_label_df["y"] == 1, "group_id"].astype(str).tolist()
neg_groups = group_label_df.loc[group_label_df["y"] == 0, "group_id"].astype(str).tolist()

y_by_group = dict(zip(group_label_df["group_id"].astype(str), group_label_df["y"].astype(int)))
outer_folds = []
for seed in OUTER_SEEDS:
    rng = np.random.default_rng(seed)
    repeat_folds = make_outer_folds(list(rng.permutation(pos_groups)), list(rng.permutation(neg_groups)))
    held_out = [g for _, te in repeat_folds for g in te]
    assert sorted(held_out) == sorted(pos_groups + neg_groups)
    assert all(not set(tr) & set(te) for tr, te in repeat_folds)
    outer_folds.extend(repeat_folds)

fold_manifest_rows = []
for i, (tr, te) in enumerate(outer_folds, start=1):
    fold_manifest_rows.append(
        {
            "outer_fold_id": i,
            "repeat_id": (i - 1) // OUTER_N_SPLITS + 1,
            "fold_in_repeat": (i - 1) % OUTER_N_SPLITS + 1,
            "split_seed": OUTER_SEEDS[(i - 1) // OUTER_N_SPLITS],
            "train_groups": "|".join(tr),
            "test_groups": "|".join(te),
            "n_train_groups": len(tr),
            "n_test_groups": len(te),
        }
    )
fold_manifest = pd.DataFrame(fold_manifest_rows)
display(fold_manifest)


X_mean_delta: (90, 88)
X_delta_only: (90, 44)
X_mfr_only: (90, 2)
N samples: 90
N groups: 15


,outer_fold_id,repeat_id,fold_in_repeat,split_seed,train_groups,test_groups,n_train_groups,n_test_groups
0,1,1,1,0,NEG03|NEG04|NEG05|NEG02|POS03|POS08|POS04|POS0...,NEG01|POS05|POS07,12,3
1,2,1,2,0,NEG01|NEG04|NEG05|NEG02|POS05|POS07|POS04|POS0...,NEG03|POS03|POS08,12,3
2,3,1,3,0,NEG01|NEG03|NEG05|NEG02|POS05|POS07|POS03|POS0...,NEG04|POS04|POS06,12,3
3,4,1,4,0,NEG01|NEG03|NEG04|NEG02|POS05|POS07|POS03|POS0...,NEG05|POS10|POS01,12,3
4,5,1,5,0,NEG01|NEG03|NEG04|NEG05|POS05|POS07|POS03|POS0...,NEG02|POS09|POS02,12,3
5,6,2,1,1,NEG01|NEG03|NEG04|NEG05|POS08|POS01|POS02|POS0...,NEG02|POS09|POS05,12,3
6,7,2,2,1,NEG02|NEG03|NEG04|NEG05|POS09|POS05|POS02|POS0...,NEG01|POS08|POS01,12,3
7,8,2,3,1,NEG02|NEG01|NEG04|NEG05|POS09|POS05|POS08|POS0...,NEG03|POS02|POS03,12,3
8,9,2,4,1,NEG02|NEG01|NEG03|NEG05|POS09|POS05|POS08|POS0...,NEG04|POS06|POS10,12,3
9,10,2,5,1,NEG02|NEG01|NEG03|NEG04|POS09|POS05|POS08|POS0...,NEG05|POS07|POS04,12,3


## Models and feature representations

In [3]:
model_specs = make_model_specs()
display(pd.DataFrame([{"model_name": d["model_name"], "simplicity_rank": d["simplicity_rank"]} for d in model_specs]))

FEATURE_SET_MAP = {
    "mean+delta": X_mean_delta_df,
    "delta-only": X_delta_only_df,
    "mfr-only": X_mfr_only_df,
}
feature_set_complexity_rank = {
    "delta-only": 0,
    "mean+delta": 1,
    "mfr-only": 2,
}


,model_name,simplicity_rank
0,Logistic Regression,0
1,SVM (RBF),1
2,Random Forest,2
3,Gradient Boosting,3
4,XGBoost,4
5,KNN,5
6,CatBoost,6


## Nested cross-validation

In [4]:
def evaluate_combination(feature_set_name, X_df, spec):

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        results_rows = []
        outer_metric_rows = []
        oof_rows = []
        repeat_metric_rows = []
        repeat_oof_rows = []
        tuning_rows = []
        outer_fit_artifacts = {}

        model_name = spec["model_name"]

        fold_scores = []
        repeat_probs = np.full((N_REPEATS, len(y)), np.nan, dtype=float)
        repeat_preds = np.full((N_REPEATS, len(y)), -1, dtype=int)
        repeat_fold_ids = np.full((N_REPEATS, len(y)), -1, dtype=int)
        fitted_models = []

        for outer_fold_id, (train_groups, test_groups) in enumerate(outer_folds, start=1):
            repeat_idx = (outer_fold_id - 1) // OUTER_N_SPLITS
            fold_in_repeat = (outer_fold_id - 1) % OUTER_N_SPLITS + 1
            X_train, train_mask = subset_rows(X_df, groups, train_groups)
            X_test, test_mask = subset_rows(X_df, groups, test_groups)
            y_train = y[train_mask]
            y_test = y[test_mask]

            inner_folds = make_inner_folds(train_groups, y_by_group, seed=RANDOM_STATE + 100 * outer_fold_id)

            best_params, best_cv_score, tuning_trace = bayes_optimize_model(
                model_name=model_name,
                estimator=spec["estimator"],
                space=spec["space"],
                X_df=X_train,
                y_arr=y_train,
                groups_arr=groups[train_mask],
                inner_folds=inner_folds,
                seed=RANDOM_STATE + 1000 * outer_fold_id + spec["simplicity_rank"],
                n_init=BAYES_N_INIT,
                n_iter=BAYES_N_ITER,
            )
            tuning_trace = tuning_trace.copy()
            tuning_trace["outer_fold_id"] = outer_fold_id
            tuning_trace["repeat_id"] = repeat_idx + 1
            tuning_trace["fold_in_repeat"] = fold_in_repeat
            tuning_trace["model_name"] = model_name
            tuning_trace["feature_set"] = feature_set_name
            tuning_rows.append(tuning_trace)

            y_pred, y_proba, fitted_pipe = fit_predict_one_split(
                model_name=model_name,
                estimator=spec["estimator"],
                params=best_params,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
            )

            assert np.all(repeat_fold_ids[repeat_idx, test_mask] == -1)
            repeat_probs[repeat_idx, test_mask] = y_proba
            repeat_preds[repeat_idx, test_mask] = y_pred
            repeat_fold_ids[repeat_idx, test_mask] = outer_fold_id

            fold_metric = compute_metrics(y_test, y_pred, y_proba)
            fold_metric.update(
                {
                    "outer_fold_id": outer_fold_id,
                    "repeat_id": repeat_idx + 1,
                    "fold_in_repeat": fold_in_repeat,
                    "model_name": model_name,
                    "feature_set": feature_set_name,
                    "n_test": int(len(y_test)),
                    "best_inner_cv_roc_auc": best_cv_score,
                    "best_params_json": json.dumps(best_params),
                }
            )
            outer_metric_rows.append(fold_metric)
            fold_scores.append(fold_metric["roc_auc"])

            fitted_models.append(
                {
                    "outer_fold_id": outer_fold_id,
                    "repeat_id": repeat_idx + 1,
                    "fold_in_repeat": fold_in_repeat,
                    "test_groups": test_groups,
                    "train_groups": train_groups,
                    "best_params": best_params,
                    "estimator": fitted_pipe,
                    "selected_columns_after_preprocessing": list(
                        fitted_pipe.named_steps["corr"].selected_columns_
                    ) if ("corr" in fitted_pipe.named_steps) else list(X_train.columns),
                }
            )

        if np.any(np.isnan(repeat_probs)) or np.any(repeat_preds < 0):
            raise RuntimeError(f"Incomplete OOF predictions for {model_name} / {feature_set_name}")

        repeat_metrics = []
        for repeat_idx in range(N_REPEATS):
            metrics = compute_metrics(y, repeat_preds[repeat_idx], repeat_probs[repeat_idx])
            repeat_metrics.append(metrics)
            repeat_metric_rows.append({
                "repeat_id": repeat_idx + 1,
                "split_seed": OUTER_SEEDS[repeat_idx],
                "model_name": model_name,
                "feature_set": feature_set_name,
                **metrics,
            })
            for idx in range(len(y)):
                repeat_oof_rows.append({
                    "sample_index": idx,
                    "group_id": groups[idx],
                    "y_true": int(y[idx]),
                    "y_pred": int(repeat_preds[repeat_idx, idx]),
                    "y_proba": float(repeat_probs[repeat_idx, idx]),
                    "model_name": model_name,
                    "feature_set": feature_set_name,
                    "repeat_id": repeat_idx + 1,
                    "outer_fold_id": int(repeat_fold_ids[repeat_idx, idx]),
                    "fold_in_repeat": int((repeat_fold_ids[repeat_idx, idx] - 1) % OUTER_N_SPLITS + 1),
                })

        fold_probs = repeat_probs.mean(axis=0)
        fold_preds = (fold_probs >= 0.5).astype(int)
        pooled = compute_metrics(y, fold_preds, fold_probs)
        result_row = {
            "model_name": model_name,
            "feature_set": feature_set_name,
            "roc_auc_mean": float(np.mean([r["roc_auc"] for r in outer_metric_rows if r["model_name"] == model_name and r["feature_set"] == feature_set_name])),
            "roc_auc_std": float(np.std([r["roc_auc"] for r in outer_metric_rows if r["model_name"] == model_name and r["feature_set"] == feature_set_name], ddof=1)),
            "pooled_roc_auc": pooled["roc_auc"],
            "pooled_pr_auc": pooled["pr_auc"],
            "pooled_accuracy": pooled["accuracy"],
            "pooled_balanced_accuracy": pooled["balanced_accuracy"],
            "pooled_f1": pooled["f1"],
            "pooled_precision": pooled["precision"],
            "pooled_recall": pooled["recall"],
            "pooled_specificity": pooled["specificity"],
            "model_simplicity_rank": spec["simplicity_rank"],
            "feature_set_complexity_rank": feature_set_complexity_rank[feature_set_name],
            "is_baseline_feature_set": feature_set_name == "mfr-only",
        }
        for metric in pooled:
            values = [m[metric] for m in repeat_metrics]
            result_row[f"repeat_{metric}_mean"] = float(np.mean(values))
            result_row[f"repeat_{metric}_std"] = float(np.std(values, ddof=1)) if N_REPEATS > 1 else 0.0
        results_rows.append(result_row)

        for idx in range(len(y)):
            oof_rows.append(
                {
                    "sample_index": idx,
                    "group_id": groups[idx],
                    "y_true": int(y[idx]),
                    "y_pred": int(fold_preds[idx]),
                    "y_proba": float(fold_probs[idx]),
                    "model_name": model_name,
                    "feature_set": feature_set_name,
                    "outer_fold_id": int(repeat_fold_ids[0, idx]) if N_REPEATS == 1 else None,
                    "outer_fold_ids": "|".join(str(int(v)) for v in repeat_fold_ids[:, idx]),
                    "n_repeats": N_REPEATS,
                }
            )

        outer_fit_artifacts[(model_name, feature_set_name)] = fitted_models

        return {
            "results_rows": results_rows, "outer_metric_rows": outer_metric_rows,
            "oof_rows": oof_rows, "repeat_metric_rows": repeat_metric_rows,
            "repeat_oof_rows": repeat_oof_rows, "tuning_rows": tuning_rows,
            "outer_fit_artifacts": outer_fit_artifacts,
        }

training_protocol = make_training_protocol(BUNDLE_PATH)
training_protocol_hash = joblib.hash(training_protocol)
combination_tasks = [(name, X_df, spec) for name, X_df in FEATURE_SET_MAP.items() for spec in model_specs]
combination_results = []
for feature_set_name, X_df, spec in combination_tasks:
    print(f"Evaluating {spec['model_name']} / {feature_set_name}")
    combination_results.append(evaluate_combination(feature_set_name, X_df, spec))

results_rows = []
outer_metric_rows = []
oof_rows = []
repeat_metric_rows = []
repeat_oof_rows = []
tuning_rows = []
outer_fit_artifacts = {}

for result in combination_results:
    results_rows.extend(result["results_rows"])
    outer_metric_rows.extend(result["outer_metric_rows"])
    oof_rows.extend(result["oof_rows"])
    repeat_metric_rows.extend(result["repeat_metric_rows"])
    repeat_oof_rows.extend(result["repeat_oof_rows"])
    tuning_rows.extend(result["tuning_rows"])
    outer_fit_artifacts.update(result["outer_fit_artifacts"])

del combination_results

results_df = pd.DataFrame(results_rows)
outer_metrics_df = pd.DataFrame(outer_metric_rows)
oof_df = pd.DataFrame(oof_rows)
repeat_metrics_df = pd.DataFrame(repeat_metric_rows)
repeat_oof_df = pd.DataFrame(repeat_oof_rows)
tuning_df = pd.concat(tuning_rows, axis=0, ignore_index=True)

display(results_df.sort_values(["pooled_roc_auc", "pooled_pr_auc"], ascending=[False, False]))
print("Finished all model × feature-set combinations.")


Evaluating Logistic Regression / mean+delta
Evaluating SVM (RBF) / mean+delta
Evaluating Random Forest / mean+delta
Evaluating Gradient Boosting / mean+delta
Evaluating XGBoost / mean+delta
Evaluating KNN / mean+delta
Evaluating CatBoost / mean+delta
Evaluating Logistic Regression / delta-only
Evaluating SVM (RBF) / delta-only
Evaluating Random Forest / delta-only
Evaluating Gradient Boosting / delta-only
Evaluating XGBoost / delta-only
Evaluating KNN / delta-only
Evaluating CatBoost / delta-only
Evaluating Logistic Regression / mfr-only
Evaluating SVM (RBF) / mfr-only
Evaluating Random Forest / mfr-only
Evaluating Gradient Boosting / mfr-only
Evaluating XGBoost / mfr-only
Evaluating KNN / mfr-only
Evaluating CatBoost / mfr-only


,model_name,feature_set,roc_auc_mean,roc_auc_std,pooled_roc_auc,pooled_pr_auc,pooled_accuracy,pooled_balanced_accuracy,pooled_f1,pooled_precision,...,repeat_balanced_accuracy_mean,repeat_balanced_accuracy_std,repeat_f1_mean,repeat_f1_std,repeat_precision_mean,repeat_precision_std,repeat_recall_mean,repeat_recall_std,repeat_specificity_mean,repeat_specificity_std
4,XGBoost,mean+delta,0.884722,0.157437,0.909444,0.953164,0.855556,0.816667,0.896000,0.861538,...,0.813889,0.026788,0.885946,0.013001,0.867337,0.021323,0.905556,0.009623,0.722222,0.050918
3,Gradient Boosting,mean+delta,0.887963,0.138015,0.891667,0.936087,0.855556,0.833333,0.892562,0.885246,...,0.811111,0.039382,0.875516,0.020310,0.873449,0.031003,0.877778,0.009623,0.744444,0.069389
2,Random Forest,mean+delta,0.865278,0.167199,0.875000,0.913385,0.833333,0.791667,0.880000,0.846154,...,0.780556,0.038490,0.871763,0.020327,0.840679,0.028237,0.905556,0.019245,0.655556,0.069389
11,XGBoost,delta-only,0.870370,0.152838,0.872778,0.931704,0.800000,0.741667,0.859375,0.808824,...,0.766667,0.044096,0.871059,0.033860,0.825678,0.023615,0.922222,0.050918,0.611111,0.050918
7,Logistic Regression,delta-only,0.807407,0.145182,0.852222,0.933545,0.733333,0.700000,0.800000,0.800000,...,0.716667,0.036324,0.801004,0.043154,0.816639,0.023637,0.788889,0.078764,0.644444,0.069389
0,Logistic Regression,mean+delta,0.876852,0.170739,0.852222,0.923214,0.788889,0.766667,0.840336,0.847458,...,0.791667,0.036324,0.844919,0.021924,0.875294,0.029296,0.816667,0.016667,0.766667,0.057735
6,CatBoost,mean+delta,0.872222,0.169308,0.832778,0.875774,0.822222,0.775000,0.873016,0.833333,...,0.783333,0.014434,0.874361,0.019698,0.841033,0.004441,0.911111,0.041944,0.655556,0.019245
10,Gradient Boosting,delta-only,0.867130,0.159071,0.832222,0.842990,0.811111,0.750000,0.868217,0.811594,...,0.741667,0.041667,0.855818,0.013123,0.812304,0.033094,0.905556,0.025459,0.577778,0.101835
9,Random Forest,delta-only,0.846296,0.168677,0.809444,0.882035,0.766667,0.691667,0.839695,0.774648,...,0.705556,0.024056,0.843261,0.006177,0.785089,0.018084,0.911111,0.009623,0.500000,0.057735
8,SVM (RBF),delta-only,0.788889,0.141853,0.803889,0.875212,0.722222,0.691667,0.789916,0.796610,...,0.680556,0.055486,0.789695,0.043747,0.785164,0.034160,0.794444,0.053576,0.566667,0.057735


Finished all model × feature-set combinations.


## Primary model selection

In [5]:
candidate_df = results_df.loc[~results_df["is_baseline_feature_set"]].copy()

primary_ranked = candidate_df.sort_values(
    by=[

        "pooled_roc_auc",
        "pooled_pr_auc",
        "pooled_balanced_accuracy",
        "roc_auc_std",
        "feature_set_complexity_rank",
        "model_simplicity_rank",
    ],
    ascending=[

               False, False, False, True, True, True],
).reset_index(drop=True)

selected_row = primary_ranked.iloc[0].copy()
selected_model_name = str(selected_row["model_name"])
selected_feature_set = str(selected_row["feature_set"])

print("Selected primary combination:")
display(pd.DataFrame([selected_row]))

best_by_feature_set = (
    results_df.sort_values(
        by=["feature_set", "pooled_roc_auc", "pooled_pr_auc", "pooled_balanced_accuracy", "roc_auc_std", "model_simplicity_rank"],
        ascending=[True, False, False, False, True, True],
    )
    .groupby("feature_set", as_index=False)
    .first()
)

best_by_model = (
    results_df.sort_values(
        by=["model_name", "pooled_roc_auc", "pooled_pr_auc", "pooled_balanced_accuracy", "roc_auc_std", "feature_set_complexity_rank"],
        ascending=[True, False, False, False, True, True],
    )
    .groupby("model_name", as_index=False)
    .first()
)

display(best_by_feature_set)
display(best_by_model)


Selected primary combination:


,model_name,feature_set,roc_auc_mean,roc_auc_std,pooled_roc_auc,pooled_pr_auc,pooled_accuracy,pooled_balanced_accuracy,pooled_f1,pooled_precision,...,repeat_balanced_accuracy_mean,repeat_balanced_accuracy_std,repeat_f1_mean,repeat_f1_std,repeat_precision_mean,repeat_precision_std,repeat_recall_mean,repeat_recall_std,repeat_specificity_mean,repeat_specificity_std
0,XGBoost,mean+delta,0.884722,0.157437,0.909444,0.953164,0.855556,0.816667,0.896,0.861538,...,0.813889,0.026788,0.885946,0.013001,0.867337,0.021323,0.905556,0.009623,0.722222,0.050918


,feature_set,model_name,roc_auc_mean,roc_auc_std,pooled_roc_auc,pooled_pr_auc,pooled_accuracy,pooled_balanced_accuracy,pooled_f1,pooled_precision,...,repeat_balanced_accuracy_mean,repeat_balanced_accuracy_std,repeat_f1_mean,repeat_f1_std,repeat_precision_mean,repeat_precision_std,repeat_recall_mean,repeat_recall_std,repeat_specificity_mean,repeat_specificity_std
0,delta-only,XGBoost,0.870370,0.152838,0.872778,0.931704,0.800000,0.741667,0.859375,0.808824,...,0.766667,0.044096,0.871059,0.033860,0.825678,0.023615,0.922222,0.050918,0.611111,0.050918
1,mean+delta,XGBoost,0.884722,0.157437,0.909444,0.953164,0.855556,0.816667,0.896000,0.861538,...,0.813889,0.026788,0.885946,0.013001,0.867337,0.021323,0.905556,0.009623,0.722222,0.050918
2,mfr-only,CatBoost,0.541204,0.178550,0.559444,0.724051,0.600000,0.516667,0.718750,0.676471,...,0.508333,0.043301,0.698597,0.023925,0.672294,0.026435,0.727778,0.034694,0.288889,0.083887


,model_name,feature_set,roc_auc_mean,roc_auc_std,pooled_roc_auc,pooled_pr_auc,pooled_accuracy,pooled_balanced_accuracy,pooled_f1,pooled_precision,...,repeat_balanced_accuracy_mean,repeat_balanced_accuracy_std,repeat_f1_mean,repeat_f1_std,repeat_precision_mean,repeat_precision_std,repeat_recall_mean,repeat_recall_std,repeat_specificity_mean,repeat_specificity_std
0,CatBoost,mean+delta,0.872222,0.169308,0.832778,0.875774,0.822222,0.775000,0.873016,0.833333,...,0.783333,0.014434,0.874361,0.019698,0.841033,0.004441,0.911111,0.041944,0.655556,0.019245
1,Gradient Boosting,mean+delta,0.887963,0.138015,0.891667,0.936087,0.855556,0.833333,0.892562,0.885246,...,0.811111,0.039382,0.875516,0.020310,0.873449,0.031003,0.877778,0.009623,0.744444,0.069389
2,KNN,mean+delta,0.814352,0.242480,0.792222,0.882290,0.755556,0.708333,0.822581,0.796875,...,0.702778,0.025459,0.816236,0.011288,0.795177,0.019691,0.838889,0.019245,0.566667,0.057735
3,Logistic Regression,delta-only,0.807407,0.145182,0.852222,0.933545,0.733333,0.700000,0.800000,0.800000,...,0.716667,0.036324,0.801004,0.043154,0.816639,0.023637,0.788889,0.078764,0.644444,0.069389
4,Random Forest,mean+delta,0.865278,0.167199,0.875000,0.913385,0.833333,0.791667,0.880000,0.846154,...,0.780556,0.038490,0.871763,0.020327,0.840679,0.028237,0.905556,0.019245,0.655556,0.069389
5,SVM (RBF),delta-only,0.788889,0.141853,0.803889,0.875212,0.722222,0.691667,0.789916,0.796610,...,0.680556,0.055486,0.789695,0.043747,0.785164,0.034160,0.794444,0.053576,0.566667,0.057735
6,XGBoost,mean+delta,0.884722,0.157437,0.909444,0.953164,0.855556,0.816667,0.896000,0.861538,...,0.813889,0.026788,0.885946,0.013001,0.867337,0.021323,0.905556,0.009623,0.722222,0.050918


## Representation comparison

In [6]:
selected_model_feature_set_results = (
    results_df.loc[results_df["model_name"] == selected_model_name]
    .copy()
    .sort_values("feature_set")
    .reset_index(drop=True)
)
display(selected_model_feature_set_results)


,model_name,feature_set,roc_auc_mean,roc_auc_std,pooled_roc_auc,pooled_pr_auc,pooled_accuracy,pooled_balanced_accuracy,pooled_f1,pooled_precision,...,repeat_balanced_accuracy_mean,repeat_balanced_accuracy_std,repeat_f1_mean,repeat_f1_std,repeat_precision_mean,repeat_precision_std,repeat_recall_mean,repeat_recall_std,repeat_specificity_mean,repeat_specificity_std
0,XGBoost,delta-only,0.870370,0.152838,0.872778,0.931704,0.800000,0.741667,0.859375,0.808824,...,0.766667,0.044096,0.871059,0.033860,0.825678,0.023615,0.922222,0.050918,0.611111,0.050918
1,XGBoost,mean+delta,0.884722,0.157437,0.909444,0.953164,0.855556,0.816667,0.896000,0.861538,...,0.813889,0.026788,0.885946,0.013001,0.867337,0.021323,0.905556,0.009623,0.722222,0.050918
2,XGBoost,mfr-only,0.527778,0.139508,0.495000,0.673589,0.666667,0.525000,0.791667,0.678571,...,0.494444,0.029266,0.767940,0.021725,0.663934,0.014540,0.911111,0.041944,0.077778,0.050918


## Save models and predictions

In [7]:
if make_training_protocol(BUNDLE_PATH) != training_protocol:
    raise RuntimeError("Training settings/data changed after fitting. Restart Kernel and Run All in 03.")
export_marker = OUT_DIR / "export_in_progress.json"
export_marker.write_text(json.dumps({"random_state": RANDOM_STATE}), encoding="utf-8")
fold_manifest.to_csv(OUT_DIR / "outer_fold_manifest.csv", index=False)

results_df = results_df.sort_values(
    by=["pooled_roc_auc", "pooled_pr_auc", "pooled_balanced_accuracy", "roc_auc_std"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

results_df.to_csv(OUT_DIR / "model_results_main.csv", index=False)
outer_metrics_df.to_csv(OUT_DIR / "outer_fold_metrics_main.csv", index=False)

oof_df.to_csv(OUT_DIR / "oof_predictions_main.csv", index=False)
repeat_metrics_df.to_csv(OUT_DIR / "repeat_metrics_main.csv", index=False)
repeat_oof_df.to_csv(OUT_DIR / "oof_predictions_by_repeat.csv", index=False)
tuning_df.to_csv(OUT_DIR / "nested_tuning_detail.csv", index=False)
best_by_feature_set.to_csv(OUT_DIR / "best_by_feature_set.csv", index=False)
best_by_model.to_csv(OUT_DIR / "best_by_model.csv", index=False)

artifact = {
    "results_df_main": results_df.copy(),
    "results_df_selected_model_feature_sets": selected_model_feature_set_results.copy(),
    "outer_fold_metrics_main": outer_metrics_df.copy(),
    "oof_predictions_main": oof_df.copy(),
    "repeat_metrics_main": repeat_metrics_df.copy(),
    "oof_predictions_by_repeat": repeat_oof_df.copy(),
    "nested_tuning_detail": tuning_df.copy(),
    "outer_fitted_models": outer_fit_artifacts,
    "selected_model_name": selected_model_name,
    "selected_feature_set": selected_feature_set,
    "selection_rule": {
        "primary_candidates": ["mean+delta", "delta-only"],
        "sort_order": [
            "pooled_roc_auc desc",
            "pooled_pr_auc desc",
            "pooled_balanced_accuracy desc",
            "roc_auc_std asc",
            "feature_set_complexity_rank asc",
            "model_simplicity_rank asc",
        ],
    },
    "outer_fold_manifest": fold_manifest.copy(),
    "feature_set_map_columns": {
        "mean+delta": feature_names_mean_delta,
        "delta-only": feature_names_delta_only,
        "mfr-only": feature_names_mfr_only,
    },
    "training_protocol": training_protocol,
    "training_protocol_hash": training_protocol_hash,
    "training_bundle_sha256": training_protocol["training_bundle_sha256"],
    "random_state": RANDOM_STATE,
    "outer_n_splits": OUTER_N_SPLITS,
    "n_repeats": N_REPEATS,
    "outer_split_seeds": OUTER_SEEDS,
    "oof_aggregation": "mean probability per sample across repeats; threshold 0.5",
}
joblib.dump(artifact, OUT_DIR / "model_artifacts.joblib", compress=3)

exported_names = [
    "model_results_main.csv", "outer_fold_metrics_main.csv", "oof_predictions_main.csv",
    "repeat_metrics_main.csv", "oof_predictions_by_repeat.csv", "nested_tuning_detail.csv",
    "best_by_feature_set.csv", "best_by_model.csv", "outer_fold_manifest.csv", "model_artifacts.joblib",
]
summary = {
    "random_state": RANDOM_STATE,
    "training_protocol_hash": training_protocol_hash,
    "training_bundle_sha256": training_protocol["training_bundle_sha256"],
    "export_sha256": {name: hashlib.sha256((OUT_DIR / name).read_bytes()).hexdigest() for name in exported_names},
    "selected_model_name": selected_model_name,
    "selected_feature_set": selected_feature_set,
    "n_combinations_evaluated": int(len(results_df)),
    "outer_n_splits": OUTER_N_SPLITS,
    "n_repeats": N_REPEATS,
    "outer_split_seeds": OUTER_SEEDS,
    "oof_aggregation": "mean probability per sample across repeats; threshold 0.5",
}
with open(OUT_DIR / "ml_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

export_marker.unlink()
print("Saved outputs to:", OUT_DIR.relative_to(PROJECT_ROOT))
print("Selected primary model:", selected_model_name)
print("Selected primary feature set:", selected_feature_set)


Saved outputs to: results/03_ml
Selected primary model: XGBoost
Selected primary feature set: mean+delta
